# 🛒 Imtiaz Super Market — Data Warehousing & Mining Project
## ETL Pipeline: Extract, Transform, Load
**Organization:** Imtiaz Super Market, Karachi  
**Dataset:** Jan 2025 – Apr 2026  
**Tools:** Python, Pandas, Snowflake, Power BI  

---

## 📦 Step 1: Extract — Load Raw Data
Loading the raw dataset generated from Imtiaz Super Market's product catalog and customer transactions.

In [2]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Load raw dataset
df = pd.read_csv(r"C:\Users\hp\Sales-Trend-Analysis-and-Customer-Segmentation-at-Imtiaz-Super-Market\data\Imtiaz_Supermarket_Dataset_2025_2026.csv")

print(f"✅ Data loaded successfully!")
print(f"   Rows    : {df.shape[0]:,}")
print(f"   Columns : {df.shape[1]}")
print(f"   Date range: {df['order_date'].min()} → {df['order_date'].max()}")

✅ Data loaded successfully!
   Rows    : 17,513
   Columns : 27
   Date range: 2025-01-01 → 2026-04-15


## 🔍 Step 2: Explore Raw Data
Quick overview of the dataset structure, columns, and data types.

In [3]:
# First look at data
df.head()

,order_id,order_date,month,month_name,year,quarter,day_of_week,customer_id,customer_name,gender,...,category,unit_price_pkr,quantity,discount_pct,discount_amount_pkr,sales_pkr,profit_pkr,profit_margin_pct,payment_method,delivery_type
0,ORD001000,2025-07-04,7,July,2025,Q3,Friday,CUST0001,Ahmed Baig,Male,...,Bakery,103,4,0,0,412,81,19.7,Cash,In-Store
1,ORD001000,2025-07-04,7,July,2025,Q3,Friday,CUST0001,Ahmed Baig,Male,...,Dairy,489,4,0,0,1956,476,24.3,Debit Card,In-Store
2,ORD001000,2025-07-04,7,July,2025,Q3,Friday,CUST0001,Ahmed Baig,Male,...,Toys & Sports,793,3,0,0,2379,493,20.7,EasyPaisa,In-Store
3,ORD001000,2025-07-04,7,July,2025,Q3,Friday,CUST0001,Ahmed Baig,Male,...,Pharmacy & Health,86,3,0,0,258,28,10.9,JazzCash,In-Store
4,ORD001001,2026-03-15,3,March,2026,Q1,Sunday,CUST0001,Ahmed Baig,Male,...,Personal Care,381,4,0,0,1524,245,16.1,Cash,In-Store


In [4]:
# Shape and data types
print(f"Dataset Shape: {df.shape}")
print(f"\nColumn Names & Data Types:")
print(df.dtypes)

Dataset Shape: (17513, 27)

Column Names & Data Types:
order_id                object
order_date              object
month                    int64
month_name              object
year                     int64
quarter                 object
day_of_week             object
customer_id             object
customer_name           object
gender                  object
age_group               object
area                    object
city                    object
loyalty_member          object
store_name              object
store_id                object
product_name            object
category                object
unit_price_pkr           int64
quantity                 int64
discount_pct             int64
discount_amount_pkr      int64
sales_pkr                int64
profit_pkr               int64
profit_margin_pct      float64
payment_method          object
delivery_type           object
dtype: object


In [5]:
# Basic statistics
df.describe()

,month,year,unit_price_pkr,quantity,discount_pct,discount_amount_pkr,sales_pkr,profit_pkr,profit_margin_pct
count,17513.000000,17513.000000,17513.000000,17513.000000,17513.000000,17513.000000,17513.000000,17513.000000,17513.000000
mean,5.572946,2025.227260,451.010449,2.491463,3.719808,41.202478,1087.326843,178.811397,16.449146
std,3.559478,0.419074,453.353458,1.121017,5.137989,100.153148,1308.972594,231.579066,4.900151
min,1.000000,2025.000000,27.000000,1.000000,0.000000,0.000000,24.000000,2.000000,7.100000
25%,2.000000,2025.000000,130.000000,1.000000,0.000000,0.000000,288.000000,43.000000,12.200000
50%,5.000000,2025.000000,324.000000,2.000000,0.000000,0.000000,628.000000,97.000000,16.400000
75%,9.000000,2025.000000,593.000000,3.000000,5.000000,38.000000,1377.000000,221.000000,20.700000
max,12.000000,2026.000000,3076.000000,4.000000,15.000000,1720.000000,12304.000000,2853.000000,25.500000


## 🔄 Step 3: Transform — Clean & Enrich Data
Fixing data types, removing duplicates, adding new features like Season and Customer Segment.

In [6]:
# Fix data types
df['order_date']       = pd.to_datetime(df['order_date'])
df['unit_price_pkr']   = df['unit_price_pkr'].astype(float)
df['sales_pkr']        = df['sales_pkr'].astype(float)
df['profit_pkr']       = df['profit_pkr'].astype(float)
df['quantity']         = df['quantity'].astype(int)

# Check missing values
missing = df.isnull().sum().sum()

# Remove duplicates
before = len(df)
df = df.drop_duplicates()
after = len(df)

# Clean text columns
df['customer_name'] = df['customer_name'].str.strip().str.title()
df['product_name']  = df['product_name'].str.strip()
df['category']      = df['category'].str.strip()
df['area']          = df['area'].str.strip()

# Remove invalid rows
df = df[df['sales_pkr']      > 0]
df = df[df['quantity']       > 0]
df = df[df['unit_price_pkr'] > 0]

print(f"✅ Data types fixed")
print(f"✅ Missing values found : {missing}")
print(f"✅ Duplicates removed   : {before - after}")
print(f"✅ Text columns cleaned")
print(f"✅ Clean rows remaining : {len(df):,}")

✅ Data types fixed
✅ Missing values found : 0
✅ Duplicates removed   : 0
✅ Text columns cleaned
✅ Clean rows remaining : 17,513


In [7]:
# Add date surrogate key
df['date_key'] = df['order_date'].dt.strftime('%Y%m%d').astype(int)

# Add Season column (Pakistan context)
def get_season(month):
    if month in [12, 1, 2]:  return 'Winter'
    elif month in [3, 4, 5]: return 'Spring/Ramazan'
    elif month in [6, 7, 8]: return 'Summer'
    else:                     return 'Autumn'

df['season'] = df['month'].apply(get_season)

print("✅ Date key added")
print(f"\nSeason Distribution:")
print(df['season'].value_counts())

✅ Date key added

Season Distribution:
season
Winter            5667
Spring/Ramazan    4933
Summer            3528
Autumn            3385
Name: count, dtype: int64


In [8]:
# Customer value segmentation
customer_sales = df.groupby('customer_id')['sales_pkr'].sum()
df['customer_total_spend'] = df['customer_id'].map(customer_sales)

def segment_customer(spend):
    if spend >= 50000:   return 'High Value'
    elif spend >= 20000: return 'Mid Value'
    else:                return 'Low Value'

df['customer_segment'] = df['customer_total_spend'].apply(segment_customer)

print("✅ Customer segments created")
print(f"\nCustomer Segment Distribution:")
print(df.drop_duplicates('customer_id')['customer_segment'].value_counts())

✅ Customer segments created

Customer Segment Distribution:
customer_segment
Mid Value     277
High Value    129
Low Value      94
Name: count, dtype: int64


## 🏗️ Step 4: Build Star Schema Tables
Creating Fact and Dimension tables for the Data Warehouse.

| Table | Type | Description |
|---|---|---|
| FactSales | Fact | All transactions with keys |
| DimCustomer | Dimension | Customer details |
| DimProduct | Dimension | Product & category info |
| DimStore | Dimension | Store branch details |
| DimTime | Dimension | Date & time attributes |

In [9]:
# DimCustomer
dim_customer = df[[
    'customer_id','customer_name','gender','age_group',
    'area','city','loyalty_member','customer_segment'
]].drop_duplicates(subset='customer_id').reset_index(drop=True)
dim_customer.insert(0, 'customer_key', range(1, len(dim_customer)+1))

print(f"✅ DimCustomer : {len(dim_customer):,} rows")
dim_customer.head(3)

✅ DimCustomer : 500 rows


,customer_key,customer_id,customer_name,gender,age_group,area,city,loyalty_member,customer_segment
0,1,CUST0001,Ahmed Baig,Male,26-35,DHA Phase 5,Karachi,No,Low Value
1,2,CUST0002,Naveed Hashmi,Male,18-25,Bahadurabad,Karachi,Yes,Mid Value
2,3,CUST0003,Kashif Qureshi,Male,56+,Tariq Road,Karachi,Yes,Mid Value


In [10]:
# DimProduct
dim_product = df[[
    'product_name','category'
]].drop_duplicates(subset='product_name').reset_index(drop=True)
dim_product.insert(0, 'product_key', range(1, len(dim_product)+1))
dim_product['product_id'] = ['PROD' + str(i).zfill(4) for i in dim_product['product_key']]

print(f"✅ DimProduct : {len(dim_product):,} rows")
dim_product.head(3)

✅ DimProduct : 112 rows


,product_key,product_name,category,product_id
0,1,Samosa 4pcs,Bakery,PROD0001
1,2,Cheddar Cheese 200g,Dairy,PROD0002
2,3,Football size 4,Toys & Sports,PROD0003


In [11]:
# DimStore
dim_store = df[[
    'store_id','store_name'
]].drop_duplicates(subset='store_id').reset_index(drop=True)
dim_store.insert(0, 'store_key', range(1, len(dim_store)+1))
dim_store['store_city']    = 'Karachi'
dim_store['store_country'] = 'Pakistan'

print(f"✅ DimStore : {len(dim_store):,} rows")
dim_store.head(6)

✅ DimStore : 6 rows


,store_key,store_id,store_name,store_city,store_country
0,1,ST006,Imtiaz Sharafabad,Karachi,Pakistan
1,2,ST002,Imtiaz Nazimabad,Karachi,Pakistan
2,3,ST003,Imtiaz DHA,Karachi,Pakistan
3,4,ST001,Imtiaz Bahadurabad,Karachi,Pakistan
4,5,ST005,Imtiaz Gulshan-e-Iqbal,Karachi,Pakistan
5,6,ST004,Imtiaz Clifton,Karachi,Pakistan


In [12]:
# DimTime
dim_time = df[[
    'date_key','order_date','month','month_name',
    'year','quarter','day_of_week','season'
]].drop_duplicates(subset='date_key').reset_index(drop=True)

print(f"✅ DimTime : {len(dim_time):,} rows")
dim_time.head(3)

✅ DimTime : 470 rows


,date_key,order_date,month,month_name,year,quarter,day_of_week,season
0,20250704,2025-07-04,7,July,2025,Q3,Friday,Summer
1,20260315,2026-03-15,3,March,2026,Q1,Sunday,Spring/Ramazan
2,20251015,2025-10-15,10,October,2025,Q4,Wednesday,Autumn


In [13]:
# FactSales — merge all dimension keys
fact_sales = df.merge(dim_customer[['customer_id','customer_key']], on='customer_id')
fact_sales = fact_sales.merge(dim_product[['product_name','product_key']], on='product_name')
fact_sales = fact_sales.merge(dim_store[['store_id','store_key']], on='store_id')

fact_sales = fact_sales[[
    'order_id','date_key','customer_key','product_key','store_key',
    'unit_price_pkr','quantity','discount_pct','discount_amount_pkr',
    'sales_pkr','profit_pkr','profit_margin_pct',
    'payment_method','delivery_type'
]]

print(f"✅ FactSales : {len(fact_sales):,} rows")
fact_sales.head(3)

✅ FactSales : 17,513 rows


,order_id,date_key,customer_key,product_key,store_key,unit_price_pkr,quantity,discount_pct,discount_amount_pkr,sales_pkr,profit_pkr,profit_margin_pct,payment_method,delivery_type
0,ORD001000,20250704,1,1,1,103.0,4,0,0,412.0,81.0,19.7,Cash,In-Store
1,ORD001000,20250704,1,2,1,489.0,4,0,0,1956.0,476.0,24.3,Debit Card,In-Store
2,ORD001000,20250704,1,3,1,793.0,3,0,0,2379.0,493.0,20.7,EasyPaisa,In-Store


## 💾 Step 5: Save Tables Locally
Saving all Star Schema tables as CSV files in the data/ folder.

In [14]:
# Save all tables to data/ folder
dim_customer.to_csv(r"C:\Users\hp\Sales-Trend-Analysis-and-Customer-Segmentation-at-Imtiaz-Super-Market\data\dim_customer.csv", index=False)
dim_product.to_csv(r"C:\Users\hp\Sales-Trend-Analysis-and-Customer-Segmentation-at-Imtiaz-Super-Market\data\dim_product.csv",   index=False)
dim_store.to_csv(r"C:\Users\hp\Sales-Trend-Analysis-and-Customer-Segmentation-at-Imtiaz-Super-Market\data\dim_store.csv",       index=False)
dim_time.to_csv(r"C:\Users\hp\Sales-Trend-Analysis-and-Customer-Segmentation-at-Imtiaz-Super-Market\data\dim_time.csv",         index=False)
fact_sales.to_csv(r"C:\Users\hp\Sales-Trend-Analysis-and-Customer-Segmentation-at-Imtiaz-Super-Market\data\fact_sales.csv",     index=False)

print("✅ All tables saved to data/ folder!")
print("\n📁 Files created:")
print("   ├── dim_customer.csv")
print("   ├── dim_product.csv")
print("   ├── dim_store.csv")
print("   ├── dim_time.csv")
print("   └── fact_sales.csv")

✅ All tables saved to data/ folder!

📁 Files created:
   ├── dim_customer.csv
   ├── dim_product.csv
   ├── dim_store.csv
   ├── dim_time.csv
   └── fact_sales.csv


## ✅ ETL Pipeline Complete!

| Step | Description | Status |
|---|---|---|
| Extract | Loaded raw dataset (17,513 rows) | ✅ Done |
| Transform | Cleaned, enriched, segmented data | ✅ Done |
| Load | Star Schema tables created & saved | ✅ Done |

**Next Step → Load tables into Snowflake Data Warehouse** ❄️

## ❄️ Step 6: Load Data into Snowflake
Connecting to Snowflake and loading all 5 Star Schema tables.

In [15]:
pip install snowflake-connector-python

Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/12.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/12.1 MB ? eta -:--:--
   - -------------------------------------- 0.5/12.1 MB 2.3 MB/s eta 0:00:06
   --- ------------------------------------ 1.0/12.1 MB 2.1 MB/s eta 0:00:06
   ---- ----------------------------------- 1.3/12.1 MB 1.9 MB/s eta 0:00:06
   ----- ---------------------------------- 1.6/12.1 MB 1.8 MB/s eta 0:00:06
   ----- ---------------------------------- 1.6/12.1 MB 1.8 MB/s eta 0:00:06
   ----- ---------------------------------- 1.6/12.1 MB 1.8 MB/s eta 0:00:06
   ----- ---------------------------------- 1.6/12.1 MB 1.8 MB/s eta 0:00:06
   ----- ---------------------------------- 1.6/12.1 MB 1.8 MB/s eta 0:00:06
   ----- ---------------------------------- 1.6/12.1 MB 1.8 MB/s eta 0:00:06
   ----- ---------------------------------- 1.6/12.1 MB 1.8 MB/s eta 0:00:06
   ---------

### ❄️ Snowflake Connection

In [16]:
import snowflake.connector
from snowflake.connector.pandas_tools import write_pandas

# Snowflake connection
conn = snowflake.connector.connect(
    user     = 'SUFIYAN003',       
    password = 'Sufiyan@780Secure!',       
    account  = 'UYZXLTU-JRB82315',       
    warehouse= 'IMTIAZ_WH',
    database = 'IMTIAZ_DW',
    schema   = 'RETAIL'
)

print("✅ Snowflake connected successfully!")

✅ Snowflake connected successfully!


### 📤 Loading Tables into Snowflake

In [17]:
# DimCustomer load
dim_customer.columns = dim_customer.columns.str.upper()
success, nchunks, nrows, _ = write_pandas(conn, dim_customer, 'DIMCUSTOMER')
print(f"✅ DimCustomer loaded: {nrows:,} rows")

✅ DimCustomer loaded: 500 rows


In [18]:
# DimProduct load
dim_product.columns = dim_product.columns.str.upper()
success, nchunks, nrows, _ = write_pandas(conn, dim_product, 'DIMPRODUCT')
print(f"✅ DimProduct loaded: {nrows:,} rows")

✅ DimProduct loaded: 112 rows


In [19]:
# DimStore load
dim_store.columns = dim_store.columns.str.upper()
success, nchunks, nrows, _ = write_pandas(conn, dim_store, 'DIMSTORE')
print(f"✅ DimStore loaded: {nrows:,} rows")

✅ DimStore loaded: 6 rows


In [21]:
# DimTime load
import pandas as pd

dim_time['ORDER_DATE'] = pd.to_datetime(dim_time['ORDER_DATE']).dt.strftime('%Y-%m-%d')

success, nchunks, nrows, _ = write_pandas(conn, dim_time, 'DIMTIME')
print(f"✅ DimTime loaded: {nrows:,} rows")

✅ DimTime loaded: 470 rows


In [22]:
# FactSales load
fact_sales.columns = fact_sales.columns.str.upper()
success, nchunks, nrows, _ = write_pandas(conn, fact_sales, 'FACTSALES')
print(f"✅ FactSales loaded: {nrows:,} rows")

✅ FactSales loaded: 17,513 rows


In [23]:
# Closing Connection
conn.close()
print("✅ Snowflake connection closed!")
print("\n🎯 All tables loaded successfully into Snowflake!")
print("   ├── DimCustomer")
print("   ├── DimProduct")
print("   ├── DimStore")
print("   ├── DimTime")
print("   └── FactSales")

✅ Snowflake connection closed!

🎯 All tables loaded successfully into Snowflake!
   ├── DimCustomer
   ├── DimProduct
   ├── DimStore
   ├── DimTime
   └── FactSales
